# TP05 : ACP, Arbres de Décision, Méthodes d'Ensemble et Pipelines sklearn
# 📘 VERSION ENSEIGNANT — Corrigé commenté

> **Usage réservé à l'enseignant.** Ce notebook contient les solutions complètes avec commentaires pédagogiques.  
> Les cellules `# 🏫 NOTE ENSEIGNANT` sont destinées à guider les discussions en séance.


### Modalités de rendu

1. Chaque étudiant doit rendre un travail individuel.
2. Renommez ce fichier selon la convention : `TP05_Nom_Prenom.ipynb`.
3. Le rendu s'effectuera via le lien de dépôt communiqué par votre enseignant.
4. Assurez-vous que votre code s'exécute sans erreur (Menu : Kernel > Restart & Run All).

### Objectifs de la séance

Ce TP a pour objectif de combiner la **réduction de dimensionnalité par ACP** avec des **méthodes de classification supervisée** sur le jeu de données MNIST (images de chiffres manuscrits).

Nous étudierons progressivement :
- L'Analyse en Composantes Principales (ACP) : théorie, visualisation, reconstruction
- La Régression sur Composantes Principales (PCR)
- Les Arbres de Décision : théorie et application
- Les méthodes d'ensemble : Random Forest et AdaBoost
- Les Pipelines sklearn et la recherche d'hyperparamètres avec GridSearchCV

### Documentation utile
- [sklearn PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [sklearn DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- [sklearn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [sklearn GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

## Organisation du TP

Ce TP contient 5 parties : 4 obligatoires + 1 bonus :

1. **ACP sur MNIST — Réduction de dimensionnalité et visualisation**
2. **Régression sur Composantes Principales (PCR)**
3. **Arbres de Décision — Classification sur MNIST**
4. **Méthodes d'ensemble — Random Forest et AdaBoost**
5. **⭐ Bonus — Pipeline complet et GridSearchCV**

## Imports globaux

Exécutez cette cellule avant de commencer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, classification_report,
    mean_squared_error, accuracy_score
)

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Chargement du jeu de données MNIST

MNIST contient **70 000 images** de chiffres manuscrits (0–9), chacune de taille **28×28 pixels** (soit 784 variables).  
Nous travaillerons sur un sous-ensemble de **10 000 exemples** pour des raisons de temps de calcul.

> 💡 Chaque ligne de `X` est une image aplatie en vecteur de 784 valeurs de pixels (0–255).

In [ ]:
# Chargement MNIST — cette cellule est fournie, exécutez-la simplement
print("Chargement de MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_full, y_full = mnist.data, mnist.target.astype(int)

# Sous-ensemble de 10 000 images pour la rapidité
idx = np.random.choice(len(X_full), size=10000, replace=False)
X, y = X_full[idx], y_full[idx]

# Normalisation (pixels dans [0,1])
X = X / 255.0

print(f"Forme de X : {X.shape}  — {X.shape[0]} images, {X.shape[1]} pixels chacune")
print(f"Classes : {np.unique(y)}")

# Visualisation de quelques exemples
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X[i].reshape(28, 28), cmap='gray_r')
    ax.set_title(str(y[i]), fontsize=8)
    ax.axis('off')
plt.suptitle("Exemples MNIST (20 premières images)", y=1.02)
plt.tight_layout()
plt.show()

---

# Partie 1 — ACP sur MNIST : Réduction de Dimensionnalité et Visualisation

## Rappel théorique

L'**Analyse en Composantes Principales (ACP)** cherche les directions de variance maximale dans les données.

Étant donné une matrice de données centrée $\tilde{X} \in \mathbb{R}^{N \times d}$, l'ACP effectue la **décomposition en valeurs propres** de la matrice de covariance :

$$\Sigma = \frac{1}{N} \tilde{X}^T \tilde{X} \in \mathbb{R}^{d \times d}$$

Les **composantes principales** sont les vecteurs propres $v_1, v_2, \ldots, v_k$ associés aux plus grandes valeurs propres $\lambda_1 \geq \lambda_2 \geq \ldots$

La **variance expliquée** par la $i$-ème composante vaut :
$$\text{VE}_i = \frac{\lambda_i}{\sum_j \lambda_j}$$

> 💡 En pratique, on n'a jamais besoin de calculer $\Sigma$ explicitement : sklearn utilise une décomposition en valeurs singulières (SVD) plus stable numériquement.

## Étape 1 — Standardisation et ACP complète

Avant toute ACP, il est crucial de **centrer** les données.  
Ici les pixels sont déjà dans [0,1] mais nous appliquons quand même un `StandardScaler`.

> ⚠️ **Note sur la standardisation des images** : Le centrage (soustraction de la moyenne) est indispensable pour l'ACP. En revanche, la **division par l'écart-type** (`with_std=True` par défaut dans `StandardScaler`) peut être problématique sur des images : certains pixels sont presque toujours noirs, leur variance est quasi nulle, et la normalisation risque d'**amplifier le bruit** dans ces zones. Dans ce TP, nous appliquons la standardisation complète par souci de simplicité, mais sachez qu'en pratique on peut se contenter du centrage seul (`StandardScaler(with_std=False)`).

In [ ]:
# Séparation train/test — fournie
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 🏫 NOTE ENSEIGNANT : StandardScaler centre ET réduit par défaut (with_std=True).
# Le centrage est indispensable pour l'ACP (sinon la 1ère CP capte juste la moyenne).
# La réduction de variance est discutable sur images (pixels quasi-noirs → variance ≈ 0 → amplification du bruit).
# On garde ici with_std=True pour simplifier, mais on peut mentionner with_std=False comme alternative.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)   # transform seulement — pas de fit !

# n_components=None conserve toutes les composantes (min(n_samples, n_features))
pca_full = PCA(n_components=None, random_state=42)
X_train_pca_full = pca_full.fit_transform(X_train_sc)

print(f"Nombre de composantes : {pca_full.n_components_}")
print(f"Variance expliquée par les 5 premières composantes : {pca_full.explained_variance_ratio_[:5].round(3)}")


## Étape 2 — Scree plot et variance cumulée

Le **scree plot** permet de choisir visuellement le nombre de composantes à conserver.  
On cherche en général le **coude** de la courbe, ou le nombre de composantes qui expliquent 90–95% de la variance.

In [ ]:
# Gauche : variance expliquée par composante (100 premières)
# Droite : variance cumulée avec seuils à 90% et 95%

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

axes[0].plot(pca_full.explained_variance_ratio_[:100], color='steelblue')
axes[0].set_xlabel("Composante principale")
axes[0].set_ylabel("Variance expliquée")
axes[0].set_title("Scree plot (100 premières composantes)")

axes[1].plot(cumvar, color='darkorange')
axes[1].axhline(0.90, color='red',   linestyle='--', label='90%')
axes[1].axhline(0.95, color='green', linestyle='--', label='95%')
axes[1].set_xlabel("Nombre de composantes")
axes[1].set_ylabel("Variance cumulée")
axes[1].set_title("Variance cumulée")
axes[1].legend()

plt.tight_layout()
plt.show()

# np.searchsorted cherche le premier indice où cumvar dépasse le seuil
n_90 = np.searchsorted(cumvar, 0.90) + 1
n_95 = np.searchsorted(cumvar, 0.95) + 1
print(f"Composantes pour 90% de variance : {n_90}")
print(f"Composantes pour 95% de variance : {n_95}")
# 🏫 NOTE ENSEIGNANT : on obtient typiquement ~87 composantes pour 90%, ~120 pour 95%.
# Ratio de compression : 784 / 87 ≈ 9x. La chute rapide du scree plot indique une forte
# structure corrélée dans les images (pixels voisins très liés).


### 📝 Observations

*Combien de composantes sont nécessaires pour capturer 90% de la variance ?  
Quel est le ratio compression obtenu par rapport aux 784 pixels d'origine ?  
Que signifie la chute rapide du scree plot pour la structure des données MNIST ?*


### 📝 Éléments de réponse (enseignant)

**Composantes pour 90% de variance** : typiquement ~87 composantes sur ce sous-ensemble MNIST.  
**Ratio de compression** : 784 / 87 ≈ **9×** — on compresse 9 fois les données sans perdre plus de 10% de l'information.  
**Interprétation du scree plot** : la chute rapide indique une forte **redondance** dans les images — les pixels voisins sont très corrélés (un pixel blanc est souvent entouré de pixels blancs). L'ACP exploite cette structure pour concentrer l'information dans peu de composantes.


## Étape 3 — Visualisation des composantes principales ("eigendigits")

Chaque composante principale peut être réinterprétée comme une **image 28×28** — on les appelle les **eigendigits**.  
Elles capturent les modes de variation les plus importants dans les images de chiffres.

In [ ]:
# pca_full.components_ : shape (n_components, 784)
# Chaque ligne est un vecteur propre → on le remet en 28×28 pour visualiser

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for ax, component in zip(axes.ravel(), pca_full.components_[:20]):
    ax.imshow(component.reshape(28, 28), cmap='RdBu_r')  # divergeant : + rouge, - bleu
    ax.axis('off')

# 🏫 NOTE ENSEIGNANT : les eigendigits capturent les modes de variation globaux.
# PC1 capte souvent la luminosité globale, PC2 l'orientation gauche/droite, etc.
# La colormap divergeante (RdBu) montre les zones où la composante est positive/négative.
plt.suptitle("Les 20 premières composantes principales — 'eigendigits'", y=1.02)
plt.tight_layout()
plt.show()


## Étape 4 — Reconstruction et perte d'information

En conservant seulement $k$ composantes, on peut **reconstruire** une approximation de l'image originale.  
La qualité de la reconstruction dépend du nombre de composantes conservées.

La reconstruction s'obtient par la **transformation inverse** :
$$\hat{X} = Z_k \cdot V_k^T + \mu$$

où $Z_k$ sont les coordonnées dans l'espace réduit et $V_k$ la matrice des $k$ composantes.

In [ ]:
n_components_list = [1, 10, 30, 50, 100, 200, 784]
fig, axes = plt.subplots(1, len(n_components_list), figsize=(16, 3))

# On prend la première image du test set
img_ref = X_test_sc[[0]]

for ax, k in zip(axes, n_components_list):
    pca_k = PCA(n_components=k, random_state=42).fit(X_train_sc)
    img_reconstructed = pca_k.inverse_transform(pca_k.transform(img_ref))
    ax.imshow(img_reconstructed.reshape(28, 28), cmap='gray_r')
    ax.set_title(f"k={k}", fontsize=9)
    ax.axis('off')

# 🏫 NOTE ENSEIGNANT : avec k=1 l'image est méconnaissable.
# À partir de k≈30-50 le chiffre est clairement lisible.
# k=784 = reconstruction parfaite (perte nulle, toutes les composantes conservées).
# C'est une bonne illustration visuelle de la perte d'information liée à la compression.
plt.suptitle("Reconstruction MNIST selon le nombre de composantes ACP", y=1.02)
plt.tight_layout()
plt.show()


## Étape 5 — Visualisation 2D de l'espace latent

En projetant les données sur les **2 premières composantes principales**, on peut visualiser la structure du jeu de données dans un plan 2D.

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
X_train_2d = pca_2d.fit_transform(X_train_sc)

plt.figure(figsize=(9, 7))
sc = plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1],
                 c=y_train, cmap='tab10', s=5, alpha=0.5)
plt.colorbar(sc, label='Chiffre')
plt.title("Projection ACP 2D — MNIST (coloré par classe)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

# 🏫 NOTE ENSEIGNANT : les classes ne sont pas bien séparées en 2D — c'est attendu,
# 2 composantes ne capturent que ~20% de la variance. On distingue néanmoins des tendances :
# 0 et 1 sont souvent bien séparés (formes très différentes).
# 4, 7 et 9 se confondent (formes similaires). Cela préfigure les erreurs de classification.


### 📝 Observations

*Les classes sont-elles bien séparées dans l'espace 2D ?  
Quels chiffres semblent les plus proches ? Pouvez-vous expliquer pourquoi ?  
Que nous dit cette visualisation sur la difficulté du problème de classification ?*


### 📝 Éléments de réponse (enseignant)

**Séparation** : partielle. Les classes 0 et 1 sont assez bien séparées (formes très différentes). Les chiffres 4, 7, 9 sont proches (traits diagonaux similaires), de même que 3 et 5.  
**Implication** : 2 composantes = ~20% de la variance → une grande partie de l'information discriminante est perdue. Ce graphique illustre *pourquoi* un classifieur linéaire 2D échouerait, et *pourquoi* on a besoin de plus de composantes.


---

# Partie 2 — Régression sur Composantes Principales (PCR)

## Rappel théorique

La **Régression sur Composantes Principales (PCR)** combine l'ACP et la régression linéaire :

1. On projette $X$ sur les $k$ premières composantes principales : $Z_k = X V_k$
2. On effectue une régression OLS sur $Z_k$ : $\hat{y} = Z_k \hat{\beta}$

**Avantages :**
- Élimine la **multicolinéarité** entre variables (les composantes sont orthogonales)
- Réduit la dimension → moins de surapprentissage
- Le nombre de composantes $k$ devient un **hyperparamètre** à valider

> ⚠️ La PCR est une méthode de **régression** (prédiction d'une valeur continue). Ici, nous allons prédire la valeur du chiffre (0–9) comme si c'était un nombre réel, puis arrondir pour obtenir une classe. C'est une simplification pédagogique — en pratique, on préfère la classification directe.

## Étape 1 — PCR : Pipeline ACP + Régression Linéaire

Nous allons comparer la PCR pour différents nombres de composantes et mesurer l'erreur quadratique moyenne (MSE).

In [ ]:
n_components_range = [1, 5, 10, 20, 30, 50, 75, 100, 150, 200]
mse_train_list = []
mse_test_list  = []

for k in n_components_range:
    pcr = Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    PCA(n_components=k, random_state=42)),
        ('reg',    LinearRegression())
    ])
    pcr.fit(X_train, y_train)  # on repart des pixels bruts non standardisés (le pipeline s'en charge)
    mse_train = mean_squared_error(y_train, pcr.predict(X_train))
    mse_test  = mean_squared_error(y_test,  pcr.predict(X_test))
    mse_train_list.append(mse_train)
    mse_test_list.append(mse_test)

# Tracé MSE
plt.figure(figsize=(9, 4))
plt.plot(n_components_range, mse_train_list, 'o-', label='Train MSE', color='steelblue')
plt.plot(n_components_range, mse_test_list,  's--', label='Test MSE',  color='tomato')
plt.xlabel("Nombre de composantes ACP (k)")
plt.ylabel("MSE")
plt.title("PCR — MSE selon le nombre de composantes")
plt.legend()
plt.tight_layout()
plt.show()

# 🏫 NOTE ENSEIGNANT : le MSE test décroît rapidement puis se stabilise.
# Au-delà de k≈50-75, ajouter des composantes n'apporte quasiment plus rien :
# les composantes supplémentaires capturent du bruit peu informatif pour la régression.


## Étape 2 — Sélection du nombre de composantes par validation croisée

In [ ]:
k_candidates = [10, 30, 50, 75, 100]
cv_mse = []

for k in k_candidates:
    pcr = Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    PCA(n_components=k, random_state=42)),
        ('reg',    LinearRegression())
    ])
    # cross_val_score retourne des scores négatifs pour les métriques de perte
    scores = cross_val_score(pcr, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    cv_mse.append(-scores.mean())  # on repasse en positif

best_k = k_candidates[np.argmin(cv_mse)]
print(f"Meilleur k (CV) : {best_k}  —  MSE CV = {min(cv_mse):.4f}")

plt.figure(figsize=(7, 4))
plt.plot(k_candidates, cv_mse, 'o-', color='purple')
plt.axvline(best_k, color='red', linestyle='--', label=f'k optimal = {best_k}')
plt.xlabel("Nombre de composantes")
plt.ylabel("MSE (validation croisée 5-fold)")
plt.title("Sélection de k par validation croisée")
plt.legend()
plt.tight_layout()
plt.show()

# 🏫 NOTE ENSEIGNANT : le k optimal est souvent autour de 50-75 sur ce sous-ensemble MNIST.
# Attention : neg_mean_squared_error → on negatie le score pour obtenir le MSE positif.
# C'est un piège classique avec cross_val_score et les métriques de perte.


## Étape 3 — Comparaison OLS brut vs PCR

In [ ]:
# OLS brut
ols = Pipeline([('scaler', StandardScaler()), ('reg', LinearRegression())])
ols.fit(X_train, y_train)
mse_ols  = mean_squared_error(y_test, ols.predict(X_test))
acc_ols  = accuracy_score(y_test, np.clip(np.round(ols.predict(X_test)), 0, 9).astype(int))

# PCR optimal
pcr_best = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=best_k, random_state=42)),
    ('reg',    LinearRegression())
])
pcr_best.fit(X_train, y_train)
mse_pcr  = mean_squared_error(y_test, pcr_best.predict(X_test))
acc_pcr  = accuracy_score(y_test, np.clip(np.round(pcr_best.predict(X_test)), 0, 9).astype(int))

print("\n=== Comparaison OLS vs PCR ===")
print(f"OLS brut      — MSE test : {mse_ols:.4f}  | Accuracy (arrondi) : {acc_ols:.4f}")
print(f"PCR (k={best_k:3d})  — MSE test : {mse_pcr:.4f}  | Accuracy (arrondi) : {acc_pcr:.4f}")

# 🏫 NOTE ENSEIGNANT : les deux approches donnent des performances similaires et médiocres
# (~55-65% d'accuracy). La régression linéaire n'est pas adaptée à la classification :
# elle prédit des valeurs continues et n'a pas de notion de décision par classe.
# Arrondir les prédictions est une astuce pédagogique, pas une pratique professionnelle.
# C'est l'occasion de motiver la régression logistique ou les arbres de décision.


### 📝 Observations

*La PCR améliore-t-elle les performances par rapport à l'OLS sur les pixels bruts ?  
Pourquoi la régression linéaire n'est-elle pas idéale pour la classification ?  
Qu'aurait-on intérêt à utiliser à la place pour une vraie tâche de classification ?*


### 📝 Éléments de réponse (enseignant)

La PCR ne surpasse pas significativement l'OLS brut en accuracy (~55-65% pour les deux). La régression linéaire n'est **pas adaptée** à la classification : elle minimise une erreur quadratique continue, pas une erreur de classification.  
**Ce qu'il faudrait utiliser** : régression logistique (ou tout autre classifieur), qui modélise directement P(Y=k|X).  
La PCR reste utile pour la régression sur des données à forte multicolinéarité — son intérêt sur MNIST est surtout pédagogique.


---

# Partie 3 — Arbres de Décision : Classification sur MNIST

## Rappel théorique

Un **arbre de décision** partitionne récursivement l'espace des features en posant des questions binaires du type $x_j \leq t$.  
À chaque nœud, on choisit la coupure qui **minimise la somme des entropies pondérées** dans les deux branches filles :

$$\sum_{v \in \{\text{gauche, droite}\}} \frac{|S_v|}{|S|} H(S_v)$$

Cela revient exactement à **maximiser le gain d'information** (ou réduction d'entropie) :

$$\text{Gain}(S, A) = H(S) - \sum_{v} \frac{|S_v|}{|S|} H(S_v)$$

Puisque $H(S)$ (entropie du nœud parent) est fixé à chaque étape, minimiser la somme pondérée des entropies filles est strictement équivalent à maximiser le gain d'information.

avec l'**entropie** $H(S) = -\sum_k p_k \log_2 p_k$ ou l'**indice de Gini** $G(S) = 1 - \sum_k p_k^2$.

> 💡 **Base du logarithme** : En théorie de l'information, l'entropie est exprimée en **base 2** (unité : bits). Certains cours utilisent le **logarithme naturel** (ln) — cela change l'unité du résultat mais **pas le choix de la coupure optimale** : diviser par ln(2) est une constante qui n'affecte pas la comparaison des gains. sklearn utilise le log naturel en interne.

**Hyperparamètres clés :**
- `max_depth` : profondeur maximale de l'arbre
- `min_samples_split` : nombre minimum d'exemples pour diviser un nœud
- `criterion` : `'gini'` ou `'entropy'`

## Étape 1 — Calcul manuel de l'entropie et du gain d'information

Avant d'utiliser sklearn, calculons ces métriques à la main sur un exemple jouet.

In [ ]:
def entropy(y):
    """Entropie de Shannon d'un vecteur d'étiquettes y."""
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    # On ignore les p == 0 (0 * log(0) = 0 par convention, mais log(0) = -inf)
    return -np.sum(p * np.log2(p + 1e-12))
    # 🏫 NOTE ENSEIGNANT : np.log2 → base 2 (bits). np.log → base naturelle.
    # Les deux donnent le même classement des coupures. Le +1e-12 évite log(0).

def gini(y):
    """Indice de Gini d'un vecteur d'étiquettes y."""
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return 1 - np.sum(p ** 2)
    # 🏫 NOTE ENSEIGNANT : Gini ∈ [0, 0.5] pour 2 classes, [0, 1-1/K] pour K classes.
    # Plus rapide à calculer que l'entropie (pas de logarithme).
    # En pratique les deux critères donnent des arbres très similaires.

def information_gain(y_parent, y_left, y_right):
    """Gain d'information d'une coupure (= réduction d'entropie)."""
    n = len(y_parent)
    n_l, n_r = len(y_left), len(y_right)
    # 🏫 NOTE ENSEIGNANT : équivalent à minimiser la somme pondérée des entropies filles,
    # puisque H(parent) est une constante à chaque nœud.
    return entropy(y_parent) - (n_l/n) * entropy(y_left) - (n_r/n) * entropy(y_right)

# Test sur un exemple jouet
y_parent = np.array([0,0,0,0,1,1,1,1])  # 4 de classe 0, 4 de classe 1 → entropie max
y_left   = np.array([0,0,0,1])
y_right  = np.array([0,1,1,1])

print(f"Entropie parent  : {entropy(y_parent):.4f}  (attendu ≈ 1.0)")
print(f"Gini parent      : {gini(y_parent):.4f}  (attendu = 0.5)")
print(f"Gain information : {information_gain(y_parent, y_left, y_right):.4f}")
# 🏫 NOTE ENSEIGNANT : le gain est faible car la coupure n'est pas parfaite
# (les deux branches restent impures). Un gain = 1.0 indiquerait une séparation parfaite.


## Étape 2 — Premier arbre sur MNIST (pixels bruts)

> 💡 Nous travaillons sur un sous-ensemble à 2 chiffres (0 et 1) pour pouvoir visualiser l'arbre.

In [ ]:
# Sous-ensemble binaire 0 vs 1 — fourni
mask_train = np.isin(y_train, [0, 1])
mask_test  = np.isin(y_test,  [0, 1])
Xb_train, yb_train = X_train_sc[mask_train], y_train[mask_train]
Xb_test,  yb_test  = X_test_sc[mask_test],   y_test[mask_test]

tree_small = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
tree_small.fit(Xb_train, yb_train)

# 🏫 NOTE ENSEIGNANT : avec max_depth=3, l'arbre est lisible et les features sélectionnées
# sont des pixels individuels. On peut demander aux étudiants : "pourquoi ce pixel ?"
# (c'est un pixel qui distingue bien la forme de 0 vs 1 — ex: zone centrale).

fig, ax = plt.subplots(figsize=(20, 6))
plot_tree(tree_small, max_depth=3, filled=True,
          feature_names=[f'px_{i}' for i in range(784)],
          class_names=['0','1'], ax=ax, fontsize=8)
plt.title("Arbre de décision — MNIST binaire (0 vs 1), max_depth=3")
plt.tight_layout()
plt.show()

print(f"Accuracy train : {tree_small.score(Xb_train, yb_train):.4f}")
print(f"Accuracy test  : {tree_small.score(Xb_test,  yb_test):.4f}")


## Étape 3 — Arbre sur 10 classes (MNIST complet) et analyse du surapprentissage

In [ ]:
depths = [1, 3, 5, 8, 10, 15, None]
acc_train_tree = []
acc_test_tree  = []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train_sc, y_train)
    acc_train_tree.append(tree.score(X_train_sc, y_train))
    acc_test_tree.append(tree.score(X_test_sc,  y_test))

depths_plot = [d if d is not None else 20 for d in depths]
labels = [str(d) if d is not None else 'None\n(complet)' for d in depths]

plt.figure(figsize=(9, 4))
plt.plot(depths_plot, acc_train_tree, 'o-', label='Train', color='steelblue')
plt.plot(depths_plot, acc_test_tree,  's--', label='Test',  color='tomato')
plt.xticks(depths_plot, labels)
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Arbre de décision — Surapprentissage selon la profondeur")
plt.legend()
plt.tight_layout()
plt.show()

# 🏫 NOTE ENSEIGNANT : le surapprentissage apparaît nettement à partir de max_depth≈8-10 :
# train accuracy → 1.0 mais test accuracy plafonne ou décroît.
# Un arbre sans limite (None) mémorise le training set (accuracy train = 1.0).
# C'est la démonstration classique variance/biais : plus de profondeur = plus de variance.


## Étape 4 — Arbre sur features ACP vs pixels bruts

In [ ]:
pca_50 = PCA(n_components=50, random_state=42)
X_train_50 = pca_50.fit_transform(X_train_sc)
X_test_50  = pca_50.transform(X_test_sc)

# Arbre sur pixels bruts
t0 = time.time()
tree_raw = DecisionTreeClassifier(max_depth=10, random_state=42)
tree_raw.fit(X_train_sc, y_train)
acc_raw = tree_raw.score(X_test_sc, y_test)
t_raw = time.time() - t0

# Arbre sur ACP
t0 = time.time()
tree_pca = DecisionTreeClassifier(max_depth=10, random_state=42)
tree_pca.fit(X_train_50, y_train)
acc_pca = tree_pca.score(X_test_50, y_test)
t_pca = time.time() - t0

print("=== Arbre de décision (max_depth=10) ===")
print(f"Pixels bruts (784 feat.) — Accuracy test : {acc_raw:.4f}  | Temps : {t_raw:.2f}s")
print(f"ACP 50 composantes       — Accuracy test : {acc_pca:.4f}  | Temps : {t_pca:.2f}s")

# 🏫 NOTE ENSEIGNANT : la réduction ACP accélère l'entraînement (moins de features à tester
# à chaque nœud) mais n'améliore pas nécessairement l'accuracy d'un arbre.
# Les arbres de décision sont robustes aux features redondantes — l'ACP n'est pas
# indispensable pour eux, contrairement aux méthodes linéaires.


### 📝 Observations

*À quelle profondeur l'arbre commence-t-il à sur-apprendre ?  
La réduction ACP améliore-t-elle l'accuracy ou seulement la vitesse ?  
Quelle est la limite fondamentale d'un seul arbre de décision ?*


### 📝 Éléments de réponse (enseignant)

Le surapprentissage devient net à partir de **max_depth ≈ 8-10** : l'accuracy train monte vers 1.0 tandis que l'accuracy test plafonne ou régresse.  
L'ACP n'améliore pas l'accuracy de l'arbre mais **réduit le temps d'entraînement** (moins de features → moins de coupures à évaluer).  
**Limite fondamentale d'un arbre** : haute variance — il mémorise le bruit du training set. C'est exactement le problème qu'adressent Random Forest et AdaBoost.


---

# Partie 4 — Méthodes d'Ensemble : Random Forest et AdaBoost

## Rappel théorique

### Bagging → Random Forest
Le **bagging** (Bootstrap AGGregatING) entraîne $T$ arbres sur des **sous-échantillons bootstrap** et agrège leurs prédictions par vote majoritaire.  
Le **Random Forest** ajoute une sélection **aléatoire d'un sous-ensemble de features** à chaque coupure, réduisant la corrélation entre arbres.

$$\hat{y} = \text{vote}\left(h_1(x), h_2(x), \ldots, h_T(x)\right)$$

### Boosting → AdaBoost
Le **boosting** entraîne les arbres **séquentiellement** : chaque arbre se concentre sur les exemples mal classés par le précédent via une réévaluation des poids.

$$F_T(x) = \sum_{t=1}^{T} \alpha_t h_t(x)$$

où $\alpha_t$ est le poids de l'arbre $t$ selon son taux d'erreur.

## Étape 1 — Random Forest sur features ACP

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_50, y_train)
acc_rf_train = rf.score(X_train_50, y_train)
acc_rf_test  = rf.score(X_test_50,  y_test)
t_rf = time.time() - t0

print(f"Random Forest — Accuracy train : {acc_rf_train:.4f}  | test : {acc_rf_test:.4f}  | Temps : {t_rf:.2f}s")

# 🏫 NOTE ENSEIGNANT : le RF devrait donner ~95-97% de test accuracy sur ce sous-ensemble.
# La différence train/test est faible grâce au bagging (réduction de variance).
# n_jobs=-1 utilise tous les cœurs disponibles — important à mentionner en TP.


## Étape 2 — Effet du nombre d'arbres (n_estimators)

In [ ]:
n_estimators_list = [1, 5, 10, 25, 50, 100, 200]
rf_acc_test = []

for n in n_estimators_list:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf_n.fit(X_train_50, y_train)
    rf_acc_test.append(rf_n.score(X_test_50, y_test))

plt.figure(figsize=(8, 4))
plt.plot(n_estimators_list, rf_acc_test, 'o-', color='green')
plt.xlabel("Nombre d'arbres (n_estimators)")
plt.ylabel("Accuracy test")
plt.title("Random Forest — Accuracy selon le nombre d'arbres")
plt.tight_layout()
plt.show()

# 🏫 NOTE ENSEIGNANT : la courbe monte rapidement puis se stabilise (loi des grands nombres).
# Au-delà de ~50-100 arbres, le gain marginal est quasi nul — mais le temps double.
# C'est le compromis accuracy/temps à discuter avec les étudiants.
# Contrairement aux arbres seuls, augmenter n_estimators ne provoque pas de surapprentissage.


## Étape 3 — AdaBoost sur features ACP

In [ ]:
t0 = time.time()
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(X_train_50, y_train)
acc_ada_train = ada.score(X_train_50, y_train)
acc_ada_test  = ada.score(X_test_50,  y_test)
t_ada = time.time() - t0

print(f"AdaBoost — Accuracy train : {acc_ada_train:.4f}  | test : {acc_ada_test:.4f}  | Temps : {t_ada:.2f}s")

# 🏫 NOTE ENSEIGNANT : AdaBoost est plus lent que RF (séquentiel, pas de parallélisation).
# L'accuracy est souvent légèrement inférieure à RF sur MNIST.
# AdaBoost avec des stumps (max_depth=1) peut être moins performant sur des données complexes.
# Mentionner que sklearn utilise SAMME.R par défaut pour la multiclasse.


## Étape 3b — Probabilités prédites : Random Forest vs AdaBoost

Un aspect révélateur des deux algorithmes est la **distribution des probabilités prédites** pour chaque classe.

- **Random Forest** : les probabilités sont souvent **concentrées** sur une classe (votes majoritaires nets), reflétant des régions de décision bien délimitées.
- **AdaBoost** : les probabilités tendent à être **plus uniformes** entre les classes — même la classe prédite n'a qu'une probabilité marginalement supérieure aux autres. Cela est particulièrement visible avec la classe majoritaire.

Cette différence de comportement en probabilités correspond à des **régions de prédiction très différentes** dans l'espace des features.

> 💡 La fonction ci-dessous vous est fournie directement — elle vous permet de visualiser concrètement ce que font les deux algorithmes en termes de sorties probabilistes.

In [ ]:
# Cellule fournie — visualisation des probabilités prédites (Random Forest vs AdaBoost)
# Choisir quelques exemples du test set
n_examples = 5
idx_examples = np.arange(n_examples)

fig, axes = plt.subplots(n_examples, 2, figsize=(12, n_examples * 2.5))
classes = np.arange(10)

for row, idx in enumerate(idx_examples):
    x = X_test_50[[idx]]
    true_label = y_test[idx]

    for col, (model, name) in enumerate([(rf, 'Random Forest'), (ada, 'AdaBoost')]):
        proba = model.predict_proba(x)[0]
        pred  = model.predict(x)[0]

        axes[row, col].bar(classes, proba, color=['tomato' if c == true_label else 'steelblue' for c in classes])
        axes[row, col].set_ylim(0, 1)
        axes[row, col].set_xticks(classes)
        axes[row, col].set_title(f"{name} — vrai:{true_label}, prédit:{pred}", fontsize=9)
        axes[row, col].set_xlabel("Classe")
        if col == 0:
            axes[row, col].set_ylabel("Probabilité")

plt.suptitle("Probabilités prédites par classe — Random Forest vs AdaBoost\n(rouge = vraie classe)", y=1.01)
plt.tight_layout()
plt.show()


## Étape 4 — Tableau comparatif de tous les modèles

In [ ]:
import pandas as pd

# On récupère les valeurs calculées précédemment
results = pd.DataFrame({
    'Modèle': ['Arbre (max_depth=10, raw)', 'Arbre (max_depth=10, ACP-50)',
               'Random Forest (100 arbres, ACP-50)', 'AdaBoost (100 estim., ACP-50)'],
    'Accuracy Test': [acc_raw, acc_pca, acc_rf_test, acc_ada_test],
    'Temps (s)':     [t_raw,   t_pca,   t_rf,        t_ada]
})

print(results.to_string(index=False))

# Meilleur modèle = Random Forest
best_model = rf
y_pred_best = best_model.predict(X_test_50)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title("Matrice de confusion — Random Forest (test)")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.tight_layout()
plt.show()

# 🏫 NOTE ENSEIGNANT : les confusions les plus fréquentes sont typiquement :
# 4↔9, 3↔5, 7↔1 — des chiffres visuellement proches.
# La matrice de confusion est bien plus informative que l'accuracy seule :
# elle montre quelles classes posent problème spécifiquement.


## Étape 5 — Courbes ROC One-vs-Rest

En cours, vous avez vu l'évaluation par **ROC One-vs-Rest (OvR)** pour la classification multiclasse.  
Pour chaque chiffre $c$, on trace la courbe ROC qui oppose :
- les vrais positifs (chiffre $c$ bien identifié)
- les faux positifs (autres chiffres classés comme $c$)

L'**AUC OvR** est la moyenne des AUC individuelles, pondérée par la fréquence de chaque classe.

Le code suivant est fourni directement pour vous permettre de visualiser ces courbes sans surcharge.

In [ ]:
# Cellule fournie — courbes ROC One-vs-Rest pour Random Forest et AdaBoost
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Binarisation des labels pour OvR
classes = np.arange(10)
y_test_bin = label_binarize(y_test, classes=classes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (model, name) in zip(axes, [(rf, 'Random Forest'), (ada, 'AdaBoost')]):
    y_score = model.predict_proba(X_test_50)
    for c in classes:
        fpr, tpr, _ = roc_curve(y_test_bin[:, c], y_score[:, c])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=1.5, label=f'Chiffre {c} (AUC={roc_auc:.2f})')
    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    ax.set_xlabel('Taux de Faux Positifs'); ax.set_ylabel('Taux de Vrais Positifs')
    ax.set_title(f'Courbes ROC OvR — {name}')
    ax.legend(loc='lower right', fontsize=7)

plt.tight_layout()
plt.show()


### 📝 Observations

*Quel modèle offre le meilleur compromis accuracy / temps ?  
Quels chiffres sont le plus souvent confondus ?  
Pourquoi le Random Forest est-il généralement plus robuste qu'un seul arbre ?  
Quelle est la différence fondamentale entre bagging et boosting ?*


### 📝 Éléments de réponse (enseignant)

**Meilleur compromis** : Random Forest — accuracy ~95-97%, entraînement parallélisable.  
**Confusions fréquentes** : 4↔9, 3↔5, 7↔1 (formes visuellement proches).  
**RF vs arbre simple** : le bagging réduit la variance — chaque arbre sur-apprend, mais la moyenne de 100 arbres indépendants converge vers un estimateur à faible variance.  
**Bagging vs Boosting** : bagging → arbres indépendants en parallèle (réduction de variance) ; boosting → arbres séquentiels en corrigeant les erreurs (réduction de biais).


---

# Partie 5 — ⭐ Bonus : Pipeline Complet et GridSearchCV

## Objectif

Dans cette partie bonus, vous allez construire un **pipeline sklearn complet** combinant :
- `StandardScaler`
- `PCA` (nombre de composantes à optimiser)
- Un classifieur (à choisir)

Et utiliser **GridSearchCV** pour tester simultanément plusieurs hyperparamètres — y compris le nombre de composantes ACP.

> 💡 Le nom des hyperparamètres dans un Pipeline suit la convention `etape__parametre` (double underscore).  
> Exemple : `pca__n_components` pour le `n_components` de l'étape nommée `'pca'`.

## Étape 1 — Construction du Pipeline

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(random_state=42)),
    ('clf',    DecisionTreeClassifier(random_state=42))
])

print(pipeline)

# 🏫 NOTE ENSEIGNANT : le Pipeline garantit qu'aucune fuite de données n'est possible
# entre les étapes — le fit de chaque étape se fait seulement sur les données d'entraînement.
# Avec GridSearchCV, le Pipeline est re-fitté intégralement à chaque fold de CV :
# les paramètres du StandardScaler et de la PCA sont appris sur le fold train seulement.


## Étape 2 — GridSearchCV sur ACP + Arbre de Décision

In [ ]:
X_gs, _, y_gs, _ = train_test_split(X_train_sc, y_train, train_size=2000, random_state=42, stratify=y_train)

param_grid = {
    'pca__n_components': [20, 50, 100],
    'clf__max_depth':    [5, 10, 15],
    'clf__criterion':   ['gini', 'entropy']
}

# scoring='roc_auc_ovr' : ROC AUC One-vs-Rest — cohérent avec le cours
# 🏫 NOTE ENSEIGNANT : roc_auc_ovr est plus discriminant qu'accuracy ici car il prend en compte
# les probabilités prédites et non juste la classe finale. Avec des classes équilibrées,
# accuracy donne des résultats similaires mais roc_auc_ovr est plus général (cf. cours).
grid_search = GridSearchCV(pipeline, param_grid, cv=3,
                           scoring='roc_auc_ovr', n_jobs=-1, verbose=1)
grid_search.fit(X_gs, y_gs)

print(f"Meilleurs paramètres : {grid_search.best_params_}")
print(f"Meilleur score CV    : {grid_search.best_score_:.4f}")

best_pipe = grid_search.best_estimator_
print(f"Accuracy test (meilleur pipeline) : {best_pipe.score(X_test_sc, y_test):.4f}")

# 🏫 NOTE ENSEIGNANT : la grille 3×3×2 = 18 combinaisons × 3 folds = 54 fits.
# Le coût est linéaire en nombre de combinaisons → favoriser RandomizedSearchCV
# pour des espaces de recherche plus larges (projet final).


## Étape 3 — Exploration libre : testez votre propre grille

Remplacez l'arbre de décision par un **Random Forest** et relancez le GridSearchCV avec votre propre grille d'hyperparamètres.  
**Contrainte** : budget maximal de **50 composantes ACP**.

In [ ]:
# Solution de référence — Random Forest avec grille réduite (n_components <= 50)
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(random_state=42)),
    ('clf',    RandomForestClassifier(random_state=42, n_jobs=-1))
])

param_grid_rf = {
    'pca__n_components': [20, 50],
    'clf__n_estimators': [50, 100],
    'clf__max_depth':    [None, 10]
}

grid_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=3,
                       scoring='roc_auc_ovr', n_jobs=-1, verbose=1)
grid_rf.fit(X_gs, y_gs)

print(f"Meilleurs paramètres RF : {grid_rf.best_params_}")
print(f"Meilleur score CV RF    : {grid_rf.best_score_:.4f}")
print(f"Accuracy test           : {grid_rf.best_estimator_.score(X_test_sc, y_test):.4f}")

# 🏫 NOTE ENSEIGNANT : le RF avec GridSearchCV devrait surpasser l'arbre de la partie précédente.
# Le meilleur paramètre est souvent n_components=50, n_estimators=100, max_depth=None.
# Pour le projet final : encourager RandomizedSearchCV avec n_iter=20-30 qui explore
# l'espace plus efficacement qu'une grille exhaustive.


### 📝 Observations

*Quelle combinaison (n_components, hyperparamètres) donne les meilleures performances ?  
La recherche sur la grille est-elle coûteuse en temps ? Pourquoi ?  
Quelles alternatives à GridSearchCV pourriez-vous envisager pour un espace de recherche plus grand ?*


### 📝 Éléments de réponse (enseignant)

Le RF avec n_components=50, n_estimators=100, max_depth=None donne typiquement les meilleurs résultats.  
**Coût de GridSearchCV** : linéaire en nombre de combinaisons × folds — vite prohibitif pour de grandes grilles.  
**Alternatives** : `RandomizedSearchCV` (sample aléatoire dans l'espace des hyperparamètres, contrôle du budget via `n_iter`), `HalvingGridSearchCV` (élagage progressif des mauvais candidats).


---

## Résumé — Points Clés

| Concept | Point clé |
|---|---|
| ACP | Réduction de 784 → quelques dizaines de composantes sans perte majeure d'information |
| Eigendigits | Les composantes principales capturent les modes de variation visuels |
| PCR | ACP + régression — contrôle la multicolinéarité, k choisi par CV |
| Arbre de décision | Interprétable mais sur-apprend facilement (max_depth critique) |
| Random Forest | Bagging d'arbres → variance réduite, plus robuste qu'un seul arbre |
| AdaBoost | Boosting séquentiel → biais réduit, sensible aux outliers |
| Pipeline sklearn | Chaîne reproductible et compatible avec GridSearchCV |
| GridSearchCV | Recherche automatique des hyperparamètres optimaux par validation croisée |
| ROC AUC OvR | Critère de scoring multiclasse : moyenne des AUC One-vs-Rest, plus informatif que l'accuracy |